## **YOLOv8 Training**

## 1. Install ultralytics
Used for YOLO training.

In [2]:
# Install ultralytics and check the environment.
!pip install -U ultralytics -q

from IPython import display
display.clear_output()

!yolo settings sync=False

import ultralytics
ultralytics.checks()

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.6/112.6 GB disk)


## 2. Clone the TUMTraffic dev-kit
Used for converting the data to YOLO format (the converter only needs numpy/scipy, already in Colab).

In [ ]:
# Clone the TUMTraffic dev-kit and add it to PYTHONPATH.
!cd /content && [ -d tum-traffic-dataset-dev-kit ] || git clone -q https://github.com/tum-traffic-dataset/tum-traffic-dataset-dev-kit.git
%env PYTHONPATH=/content/tum-traffic-dataset-dev-kit

## 3. Create the zips folder

In [ ]:
# Create the folder that holds the training zips.
!mkdir -p /content/raw_zips

Upload manually into `raw_zips` the zips of **all** training sets: `R0_S0` (contains the accident) and `R0_S1`, `R0_S2` (no accidents, used as negative examples).

## 4. Unzip the archives

In [20]:
# Unzip every archive from raw_zips into raw/<name>.
!mkdir -p /content/raw
!for z in /content/raw_zips/*.zip; do n=$(basename "$z" .zip); [ -d "/content/raw/$n" ] || unzip -q "$z" -d "/content/raw/$n"; done
!ls /content/raw

a9_dataset_r00_s00  a9_dataset_r00_s01


## Pipeline

- **4** Upload the zips of the training sets and unzip them (A9 structure: `<set>/_calibration`, `_images`, `_labels`).
- **5** The A9 dev-kit (`conversion_openlabel_to_yolo.py`, patched) converts the labels to YOLO format; the code adds the `accident` class as the **union of the boxes of the involved vehicles** (larger than a single car).
- **6-7** YOLO dataset, time-based split, YOLOv8 training.
- **10** On a new zip (untrained sets): extraction -> YOLO conversion -> detection; the predicted `accident` box is expanded until it also covers the other involved vehicles.

**Warning:** A9 does not contain an "accident" class in its labels. It is inferred by the code through heuristics (Step 5), so check the diagnostic and visualization cells visually before you train.

## 5. Custom A9 R0 → YOLO converter + `accident` class (method B)

`conversion_openlabel_to_yolo.py` **crashes on R0** as it is in the repo: the R0 branch calls `write_line(...)` without `output_folder_path_labels` (hence the empty `.txt` files), divides already-normalized coordinates by 1920/1200 a second time, uses COCO IDs that ignore `van`/`trailer`, and imports `vis_utils` (which requires `open3d`). The cell below **keeps** it and applies 5 small replacements to the copy in `/content` (`DEVKIT_PATCHES`); our code adds the `accident` class on top of what it writes. Sets that are NOT in `ACCIDENT_SETS` (e.g. R0_S1, R0_S2) are treated as **negatives**: their object labels are kept, but the `accident` class is never added to them (the heuristic rules no longer run on them, so they cannot produce false positives in the training data).

**Accident definition (method B).** We add a new class, `accident`, next to the object classes:
1. Take the 3D boxes from the labels (top view: `length × width` rectangles, rotated by `rotationYaw`). They do not depend on perspective, so vehicles that merely overlap in the image do not count.
2. **Contact** = two objects whose rectangles intersect (a truck + its attached trailer are excluded).
3. The **accident** starts at the first frame in which the contact lasts at least `MIN_CONTACT_FRAMES` consecutive frames.
4. From there on, the involved vehicles are tracked frame by frame (1:1 matching with the nearest object). In each frame they get, besides their normal box, an additional `accident` box = the union of their 2D boxes.
5. Frames before the impact stay without `accident`, so the model also learns what normal traffic looks like.

**For R0 labels (without `box3d`)** the involved-vehicles rule applies: take the vehicles that are standing still on the roadway (an area through which moving traffic passes) and group them by proximity. In each group, those that stopped **first** (within the first `EARLY_S` seconds, including those already stopped from the first frame) are the involved ones, and those that stopped later are queue and are not labeled. A group is declared an accident if it has at least `MIN_CUES` out of 4 cues: abnormal cuboid pose relative to the flow in the area (orientation / aspect), sudden stop, several vehicles stopped early, traffic stopped behind. A damage-type attribute in the labels decides directly. Cuboids are rigid, so real body deformation does not appear in the labels; the abnormal pose is only a cue.

The parameters are in the first cell below. Check the result visually in the cell after the conversion, before you train.

In [ ]:
# Paths, class list and imports for converting the A9 labels to YOLO format with an added `accident` class.
import os, re, glob, json, math, shutil
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
from PIL import Image

RAW         = '/content/raw'
YOLO_LABELS = '/content/yolo_labels'
OUT         = '/content/tumtraf_yolo'

CLASSES = ['car', 'truck', 'trailer', 'van', 'motorcycle', 'bus', 'pedestrian', 'bicycle',
           'emergency_vehicle', 'other', 'license_plate_location', 'accident']
CLS = {n: i for i, n in enumerate(CLASSES)}
SKIP = {'license_plate_location'}
CONTACT_CLASSES = set(CLASSES) - {'other', 'accident', 'license_plate_location'}

# Tunable parameters of the accident definition (contact, stopped-vehicle, involved-vehicles and persistent-deformation rules).
CONTACT_MARGIN     = 0.0
MIN_CONTACT_FRAMES = 3
MAX_MISS           = 5
MAX_BOX_AREA       = 0.5

STOP_SPEED    = 1.0
MOVE_SPEED    = 8.0
MIN_STOP_S    = 2.0
ACCIDENT_RULE = 'both'
STOP_REL = 0.10
MOVE_REL = 0.50
USE_IDS  = None

EARLY_S      = 2.0
HEAD_TOL     = 25.0
ASPECT_TOL   = 0.35
CLUSTER_K    = 2.5
ABRUPT_S     = 1.0
MIN_CUES     = 2
OVL_TOL      = 0.35
OVL_FRAMES   = 3
OVL_WIN_S    = 1.5

USE_DEFORM    = True
DEF_S         = 1.0
DEF_GAP       = 2
DEF_SMOOTH_S  = 0.5
DEF_HEAD_TOL  = 30.0
DEF_ASP_TOL   = 0.40
DEF_MIN_DIAG  = 80.0
DEF_HOLD_S    = 1.5
DEF_USE_ASPECT = False
DEF_PRE_S     = 0.6
DEF_MAX_CLUSTERS = 1
MIN_FLOW     = 6
ATTR_RE      = r'damag|deform|crash|accident|collision|unfall|schaden'
ACCIDENT_IDS = []
ACCIDENT_SETS = ['r00_s00']

# File and image lookup helpers.
RAW_CATS = Counter()


def _set_id(name):
    m = re.search(r'r0*(\d+)[_\-]?s0*(\d+)', name.lower())
    return (int(m[1]), int(m[2])) if m else None


def is_accident_set(set_):
    sid = _set_id(set_)
    return any(k.lower() in set_.lower() or (sid is not None and sid == _set_id(k)) for k in ACCIDENT_SETS)


IMG_EXT = ('.png', '.jpg', '.jpeg')


def set_cam_of(P, markers):
    parts = P.parts
    for i in range(len(parts) - 2, 0, -1):
        if parts[i] in markers:
            return parts[i - 1], (parts[i + 1] if i + 2 < len(parts) else 'camera')
    return None

def parse_ts(stem):
    nums = re.findall(r'\d+', stem)
    if len(nums) >= 2 and len(nums[0]) >= 9:
        return float(f'{nums[0]}.{nums[1]}')
    return None


_IMG, _SIZE = {}, {}
def image_index():
    if not _IMG:
        for f in glob.glob(f'{RAW}/**/*.*', recursive=True):
            if f.lower().endswith(IMG_EXT):
                P = Path(f)
                sc = set_cam_of(P, {'_images', 'images'})
                if sc is None and len(P.parts) >= 4:
                    sc = (P.parts[-4], P.parts[-2])
                if sc:
                    _IMG[(sc[0], sc[1], P.stem)] = f
                    _IMG.setdefault((sc[0], None, P.stem), f)
    return _IMG


def img_path(set_, cam, stem):
    idx = image_index()
    return idx.get((set_, cam, stem)) or idx.get((set_, None, stem))


def image_size(set_, cam, stem):
    path = img_path(set_, cam, stem)
    if path is None:
        return (1920, 1200)
    if (set_, cam) not in _SIZE:
        _SIZE[(set_, cam)] = Image.open(path).size
    return _SIZE[(set_, cam)]


# Label parsing: turns each JSON label file into a list of objects with box, class, 3D footprint and cuboid pose.
def _norm_box(x1, y1, x2, y2):
    x1, y1, x2, y2 = (min(max(float(v), 0.0), 1.0) for v in (x1, y1, x2, y2))
    return (x1, y1, x2, y2) if (x2 - x1) > 1e-4 and (y2 - y1) > 1e-4 else None


def _px(box, W, H):
    x1, y1, x2, y2 = box
    return dict(ctr=((x1 + x2) / 2 * W, (y1 + y2) / 2 * H), diag=math.hypot((x2 - x1) * W, (y2 - y1) * H),
                cn=((x1 + x2) / 2, (y1 + y2) / 2))


def _attr_hit(attrs):
    def flat(x, key=''):
        if isinstance(x, dict):
            for k, v in x.items():
                yield from flat(v, str(k))
        elif isinstance(x, list):
            for v in x:
                yield from flat(v, key)
        else:
            yield key, str(x).lower()
    off = {'false', 'none', 'no', '0', '', 'null', '0.0'}
    for k, v in flat(attrs or {}):
        if re.search(ATTR_RE, v) or (re.search(ATTR_RE, k.lower()) and v not in off):
            return True
    return False


def _r0_shape(lab, box, W, H, pix):
    out = dict(head=None, attr_hit=_attr_hit(lab.get('attributes')),
               asp=math.log(max((box[2] - box[0]) * W, 1e-6) / max((box[3] - box[1]) * H, 1e-6)))
    try:
        b = lab['box3d_projected']
        sc = np.array([1.0, 1.0]) if pix else np.array([W, H])
        fr = np.mean([b['bottom_left_front'], b['bottom_right_front']], axis=0) * sc
        bk = np.mean([b['bottom_left_back'], b['bottom_right_back']], axis=0) * sc
        v = fr - bk
        if math.hypot(*v) > 2.0:
            out['head'] = math.degrees(math.atan2(v[1], v[0])) % 180.0
    except Exception:
        pass
    return out


def _cls(cat):
    cat = str(cat).lower()
    RAW_CATS[cat] += 1
    if cat in SKIP:
        return None
    return cat if (cat in CLS and cat != 'accident') else 'other'


def load_objects(path, wh):
    W, H = wh
    with open(path) as f:
        data = json.load(f)
    out, bad = [], 0
    if 'openlabel' in data:
        ods = [(oid, o['object_data']) for fr in data['openlabel']['frames'].values()
               for oid, o in fr['objects'].items()]
        for oid, od in ods:
            try:
                cls = _cls(od['type'])
                if cls is None:
                    continue
                if 'bbox' in od:
                    xc, yc, w, h = od['bbox'][0]['val']
                    if max(xc, yc, w, h) > 1.5:
                        xc, yc, w, h = xc / W, yc / H, w / W, h / H
                    box = _norm_box(xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)
                else:
                    pts = np.array([k['point2d']['val'][:2] for k in
                                    od['keypoints_2d']['attributes']['points2d']['val']], float)
                    if pts.max() > 1.5:
                        pts = pts / np.array([W, H])
                    box = _norm_box(*pts.min(0), *pts.max(0))
                if box is None:
                    continue
                o3 = dict(c3d=None, size=None, yaw=0.0)
                cub = od.get('cuboid', {}).get('val')
                if cub:
                    qx, qy, qz, qw = cub[3:7]
                    o3 = dict(c3d=(cub[0], cub[1]), size=(cub[7], cub[8]),
                              yaw=math.atan2(2 * (qw * qz + qx * qy), 1 - 2 * (qy * qy + qz * qz)))
                out.append(dict(cls=cls, box=box, oid=oid, **_px(box, W, H), **o3))
            except Exception:
                bad += 1
    else:
        for lab in data.get('labels', []):
            try:
                cls = _cls(lab['category'])
                if cls is None:
                    continue
                pts = np.array([v[:2] for v in lab['box3d_projected'].values()], float)
                pix = bool(pts.max() > 1.5)
                if pix:
                    pts = pts / np.array([W, H])
                box = _norm_box(*pts.min(0), *pts.max(0))
                if box is None:
                    continue
                o3 = dict(c3d=None, size=None, yaw=0.0)
                try:
                    b3 = lab['box3d']
                    o3 = dict(c3d=(float(b3['location']['x']), float(b3['location']['y'])),
                              size=(float(b3['dimension']['length']), float(b3['dimension']['width'])),
                              yaw=float(b3['orientation']['rotationYaw']))
                except Exception:
                    pass
                out.append(dict(cls=cls, box=box, oid=lab.get('id'), **_px(box, W, H), **_r0_shape(lab, box, W, H, pix), **o3))
            except Exception:
                bad += 1
    return out, bad


# Contact rule: intersecting 3D footprints mark an accident, tracked from the first sustained contact.
def _corners(o, grow):
    (x, y), (l, w), yaw = o['c3d'], o['size'], o['yaw']
    c, s = math.cos(yaw), math.sin(yaw)
    hl, hw = l / 2 + grow, w / 2 + grow
    loc = np.array([[-hl, -hw], [hl, -hw], [hl, hw], [-hl, hw]])
    return loc @ np.array([[c, s], [-s, c]]) + np.array([x, y])


def _overlap(a, b):
    for poly in (a, b):
        for i in range(4):
            e = poly[(i + 1) % 4] - poly[i]
            ax = np.array([-e[1], e[0]])
            pa, pb = a @ ax, b @ ax
            if pa.max() < pb.min() or pb.max() < pa.min():
                return False
    return True


def contact_set(objs):
    idx = [i for i, o in enumerate(objs) if o['cls'] in CONTACT_CLASSES and o['c3d'] and o['size']]
    hit = set()
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            A, B = objs[idx[a]], objs[idx[b]]
            if {A['cls'], B['cls']} == {'truck', 'trailer'}:
                continue
            if math.dist(A['c3d'], B['c3d']) > (math.hypot(*A['size']) + math.hypot(*B['size'])) / 2 + CONTACT_MARGIN:
                continue
            g = CONTACT_MARGIN / 2
            if _overlap(_corners(A, g), _corners(B, g)):
                hit |= {idx[a], idx[b]}
    return hit


def accident_boxes(frames, dt):
    contacts = [contact_set(objs) for objs in frames]
    n = MIN_CONTACT_FRAMES
    t0 = next((t for t in range(len(frames) - n + 1)
               if all(contacts[t + k] for k in range(n))), None)
    if t0 is None:
        return {}, None
    radius = min(15.0, max(3.0, 30.0 * dt))
    tracks, result = [], {}
    for t in range(t0, len(frames)):
        objs = frames[t]
        pairs = sorted((math.dist(tr['pos'], o['c3d']), ti, oi)
                       for ti, tr in enumerate(tracks)
                       for oi, o in enumerate(objs)
                       if o['c3d'] and o['cls'] in CONTACT_CLASSES
                       and math.dist(tr['pos'], o['c3d']) <= radius)
        claimed, used = {}, set()
        for _, ti, oi in pairs:
            if ti not in claimed and oi not in used:
                claimed[ti] = oi
                used.add(oi)
        new = []
        for ti, tr in enumerate(tracks):
            if ti in claimed:
                new.append(dict(pos=objs[claimed[ti]]['c3d'], miss=0, oi=claimed[ti]))
            elif tr['miss'] < MAX_MISS:
                new.append(dict(pos=tr['pos'], miss=tr['miss'] + 1, oi=None))
        seen = {tr['oi'] for tr in new if tr['oi'] is not None}
        for oi in contacts[t]:
            if oi not in seen:
                new.append(dict(pos=objs[oi]['c3d'], miss=0, oi=oi))
                seen.add(oi)
        tracks = new
        if not tracks:
            break
        if seen:
            bx = [objs[oi]['box'] for oi in seen]
            box = (min(b[0] for b in bx), min(b[1] for b in bx),
                   max(b[2] for b in bx), max(b[3] for b in bx))
            if (box[2] - box[0]) * (box[3] - box[1]) <= MAX_BOX_AREA:
                result[t] = box
    return result, t0


# Object tracking (3D or 2D) and speed estimation, used by the stopped-vehicle and involved-vehicles rules.
def build_tracks(frames, dt):
    radius = min(15.0, max(2.0, 40.0 * dt))
    tracks, active = [], []
    for t, objs in enumerate(frames):
        cand = sorted((math.dist(tr['pos'], o['c3d']), ti, oi)
                      for ti, tr in enumerate(active)
                      for oi, o in enumerate(objs)
                      if o['c3d'] and o['cls'] in CONTACT_CLASSES
                      and math.dist(tr['pos'], o['c3d']) <= radius)
        claimed, used = {}, set()
        for _, ti, oi in cand:
            if ti not in claimed and oi not in used:
                claimed[ti] = oi
                used.add(oi)
        nxt = []
        for ti, tr in enumerate(active):
            if ti in claimed:
                tr['xy'][t] = objs[claimed[ti]]['c3d']
                tr['oi'][t] = claimed[ti]
                tr['pos'], tr['miss'] = tr['xy'][t], 0
                nxt.append(tr)
            elif tr['miss'] < MAX_MISS:
                tr['miss'] += 1
                nxt.append(tr)
        for oi, o in enumerate(objs):
            if oi not in used and o['c3d'] and o['cls'] in CONTACT_CLASSES:
                tr = dict(xy={t: o['c3d']}, oi={t: oi}, pos=o['c3d'], miss=0)
                tracks.append(tr)
                nxt.append(tr)
        active = nxt
    return tracks


def track_speeds(tr, dt):
    K = max(1, round(1.0 / dt))
    out = {}
    for t in sorted(tr['xy']):
        if t - K in tr['xy']:
            d = math.dist(tr['xy'][t], tr['xy'][t - K])
            if 'dg' in tr:
                d /= max((tr['dg'][t] + tr['dg'][t - K]) / 2, 1.0)
            out[t] = d / (K * dt)
    return out


def stop_start(tr, dt):
    K = max(1, round(1.0 / dt))
    need = max(1, round(MIN_STOP_S / dt) - K + 1)
    first = min(tr['xy'])
    lo, hi = (STOP_REL, MOVE_REL) if 'dg' in tr else (STOP_SPEED, MOVE_SPEED)
    moved, run = False, 0
    for t, v in track_speeds(tr, dt).items():
        if v >= hi:
            moved, run = True, 0
        elif moved and v < lo:
            run += 1
            if run >= need:
                return max(first, t - need + 1 - K)
        else:
            run = 0
    return None


def ids_persistent(frames):
    last, jumps = {}, []
    for t, objs in enumerate(frames):
        cur = {}
        for o in objs:
            if o['cls'] in CONTACT_CLASSES and o.get('oid') is not None:
                cur[o['oid']] = o
                p = last.get(o['oid'])
                if p is not None:
                    jumps.append(math.dist(p['ctr'], o['ctr']) / max(o['diag'], 1.0))
        last = cur
    return len(jumps) >= 100 and float(np.median(jumps)) < 1.0


def build_tracks_2d(frames, use_ids):
    tracks = []
    if use_ids:
        by = {}
        for t, objs in enumerate(frames):
            for oi, o in enumerate(objs):
                if o['cls'] in CONTACT_CLASSES and o.get('oid') is not None:
                    tr = by.setdefault(o['oid'], dict(xy={}, dg={}, oi={}, oid=o['oid']))
                    if t not in tr['xy']:
                        tr['xy'][t], tr['dg'][t], tr['oi'][t] = o['ctr'], o['diag'], oi
        return list(by.values())
    active = []
    for t, objs in enumerate(frames):
        cand = sorted((math.dist(tr['pos'], o['ctr']) / max(tr['diag'], 1.0), ti, oi)
                      for ti, tr in enumerate(active)
                      for oi, o in enumerate(objs)
                      if o['cls'] in CONTACT_CLASSES
                      and math.dist(tr['pos'], o['ctr']) <= max(tr['diag'], 1.0))
        claimed, used = {}, set()
        for _, ti, oi in cand:
            if ti not in claimed and oi not in used:
                claimed[ti] = oi
                used.add(oi)
        nxt = []
        for ti, tr in enumerate(active):
            if ti in claimed:
                o = objs[claimed[ti]]
                tr['xy'][t], tr['dg'][t], tr['oi'][t] = o['ctr'], o['diag'], claimed[ti]
                tr['pos'], tr['diag'], tr['miss'] = o['ctr'], o['diag'], 0
                nxt.append(tr)
            elif tr['miss'] < MAX_MISS:
                tr['miss'] += 1
                nxt.append(tr)
        for oi, o in enumerate(objs):
            if oi not in used and o['cls'] in CONTACT_CLASSES:
                tr = dict(xy={t: o['ctr']}, dg={t: o['diag']}, oi={t: oi}, pos=o['ctr'], diag=o['diag'], miss=0)
                tracks.append(tr)
                nxt.append(tr)
        active = nxt
    return tracks


def tracks_for(frames, dt):
    if any(o['c3d'] for f in frames for o in f if o['cls'] in CONTACT_CLASSES):
        return build_tracks(frames, dt), '3D'
    use_ids = ids_persistent(frames) if USE_IDS is None else USE_IDS
    return build_tracks_2d(frames, use_ids), ('2D, tracking by id' if use_ids else '2D, tracking by matching')


def _union(boxes):
    return (min(b[0] for b in boxes), min(b[1] for b in boxes),
            max(b[2] for b in boxes), max(b[3] for b in boxes))


# Stopped-vehicle rule: vehicles that moved and then stopped.
def _stationary_simple(frames, dt):
    inv = [(tr, s) for tr in tracks_for(frames, dt)[0] if (s := stop_start(tr, dt)) is not None]
    if not inv:
        return {}, None
    per_frame = defaultdict(list)
    for tr, s in inv:
        for t, oi in tr['oi'].items():
            if t >= s:
                per_frame[t].append(frames[t][oi]['box'])
    out = {}
    for t, bx in per_frame.items():
        box = _union(bx)
        if (box[2] - box[0]) * (box[3] - box[1]) <= MAX_BOX_AREA:
            out[t] = box
    return out, min(s for _, s in inv)


# Involved-vehicles rule: groups of vehicles stopped on the roadway, scored by cues such as abnormal pose, sudden stop and overlap.
GX, GY = 8, 12


def _ang_diff(a, b):
    d = abs(a - b) % 180.0
    return min(d, 180.0 - d)


def _circ_mean(angles):
    sn = sum(math.sin(math.radians(2 * a)) for a in angles)
    cs = sum(math.cos(math.radians(2 * a)) for a in angles)
    return (math.degrees(math.atan2(sn, cs)) / 2.0) % 180.0


def _cell(cn):
    return (min(GX - 1, int(cn[0] * GX)), min(GY - 1, int(cn[1] * GY)))


def _flow_field(frames, tracks, dt):
    cells = defaultdict(list)
    for tr in tracks:
        for t, v in track_speeds(tr, dt).items():
            if v >= MOVE_REL:
                o = frames[t][tr['oi'][t]]
                if o.get('head') is not None:
                    cells[_cell(o['cn'])].append((o['head'], o['asp'], o['cls']))
    return cells


def _flow_at(cells, cn, cls):
    cx, cy = _cell(cn)
    near = [x for dx in (-1, 0, 1) for dy in (-1, 0, 1) for x in cells.get((cx + dx, cy + dy), [])]
    pool = cells.get((cx, cy), [])
    if len(pool) < MIN_FLOW:
        pool = near
    if len(near) < MIN_FLOW or len(pool) < MIN_FLOW:
        return None
    same = [p[1] for p in pool if p[2] == cls]
    return _circ_mean([p[0] for p in pool]), float(np.median(same if len(same) >= 4 else [p[1] for p in pool]))


def _stationary_info(tr, dt):
    K = max(1, round(1.0 / dt))
    need = max(1, round(MIN_STOP_S / dt) - K + 1)
    sp = track_speeds(tr, dt)
    ts = sorted(sp)
    run = []
    for t in reversed(ts):
        if sp[t] < STOP_REL:
            run.append(t)
        else:
            break
    if len(run) < need:
        return None
    run_start = run[-1]
    moved = any(sp[t] >= MOVE_REL for t in ts if t < run_start)
    first = min(tr['xy'])
    return (max(first, run_start - K) if moved else first), moved, run_start


def _last_move(tr, dt, s):
    K2 = max(1, round(0.5 / dt))
    last_move = None
    for t in sorted(tr['xy']):
        if t - K2 in tr['xy']:
            d = math.dist(tr['xy'][t], tr['xy'][t - K2]) / max((tr['dg'][t] + tr['dg'][t - K2]) / 2, 1.0) / (K2 * dt)
            if d >= MOVE_REL and t < s + 2 * K2:
                last_move = t
    return last_move


def _abrupt(tr, dt, s):
    lm = _last_move(tr, dt, s)
    return lm is not None and (s - lm) * dt <= ABRUPT_S


def _box_ioa(a, b):
    ix = min(a[2], b[2]) - max(a[0], b[0])
    iy = min(a[3], b[3]) - max(a[1], b[1])
    if ix <= 0 or iy <= 0:
        return 0.0
    m = min((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1]))
    return ix * iy / m if m > 0 else 0.0


def _stop_overlap(c, partners, frames, dt):
    if not c['moved']:
        return None
    w = max(1, round(OVL_WIN_S / dt))
    tr, run, first = c['tr'], 0, None
    for t in range(c['s'] - w, c['s'] + w + 1):
        oi = tr['oi'].get(t)
        hit = False
        if oi is not None and 0 <= t < len(frames):
            b = frames[t][oi]['box']
            for d in partners:
                dj = d['tr']['oi'].get(t)
                if d is not c and d['s'] <= t and dj is not None and _box_ioa(b, frames[t][dj]['box']) >= OVL_TOL:
                    hit = True
                    break
        if hit:
            if run == 0:
                first = t
            run += 1
            if run >= OVL_FRAMES:
                return first
        else:
            run, first = 0, None
    return None


def analyze_involved(frames, dt):
    tracks, mode = tracks_for(frames, dt)
    cells = _flow_field(frames, tracks, dt)
    cands = []
    for tr in tracks:
        info = _stationary_info(tr, dt)
        if info is None:
            continue
        s, moved, _ = info
        objs = [frames[t][tr['oi'][t]] for t in sorted(tr['oi']) if t >= s]
        if not objs:
            continue
        cn = (float(np.median([o['cn'][0] for o in objs])), float(np.median([o['cn'][1] for o in objs])))
        hs = [o['head'] for o in objs if o.get('head') is not None]
        flow = _flow_at(cells, cn, objs[0]['cls'])
        hd = _ang_diff(_circ_mean(hs), flow[0]) if (flow and hs) else None
        ad = abs(float(np.median([o['asp'] for o in objs])) - flow[1]) if flow else None
        c = dict(tr=tr, s=s, moved=moved, oid=tr.get('oid'), cls=objs[0]['cls'], cn=cn,
                 ctr=(float(np.median([o['ctr'][0] for o in objs])), float(np.median([o['ctr'][1] for o in objs]))),
                 diag=float(np.median([o['diag'] for o in objs])), head_diff=hd, asp_diff=ad,
                 on_road=flow is not None, hit=any(o.get('attr_hit') for o in objs),
                 abrupt=moved and _abrupt(tr, dt, s))
        c['anom'] = bool((hd is not None and hd > HEAD_TOL) or (ad is not None and ad > ASPECT_TOL))
        lm = _last_move(tr, dt, s) if c['abrupt'] else None
        c['imp'] = min(s, lm) if lm is not None else s
        c['role'] = 'ignored'
        if c['on_road'] or c['hit']:
            cands.append(c)

    parent = list(range(len(cands)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i in range(len(cands)):
        for j in range(i + 1, len(cands)):
            if math.dist(cands[i]['ctr'], cands[j]['ctr']) < CLUSTER_K * (cands[i]['diag'] + cands[j]['diag']) / 2:
                parent[find(i)] = find(j)
    groups = defaultdict(list)
    for i, c in enumerate(cands):
        groups[find(i)].append(c)

    clusters, involved = [], []
    for members in groups.values():
        for c in members:
            c['ovl'] = _stop_overlap(c, members, frames, dt)
            if c['ovl'] is not None:
                c['imp'] = min(c['imp'], c['ovl'])
        movers = [c for c in members if c['moved']]
        trig = [c for c in movers if c['hit'] or c['abrupt'] or c['anom'] or c['ovl'] is not None]
        t_on = min(c['imp'] for c in (trig or movers or members))
        lim = t_on + round(EARLY_S / dt)
        early = [c for c in members if c['s'] <= lim]
        late = [c for c in members if c['s'] > lim]
        cues = dict(abnormal_pose=any(c['anom'] for c in early), abrupt_stop=any(c['abrupt'] for c in early),
                    several_stopped_early=len(early) >= 2 and any(c['moved'] for c in early),
                    traffic_stopped_behind=len(late) >= 1,
                    overlap_at_stop=any(c['ovl'] is not None for c in early))
        ok = any(c['hit'] for c in members) or sum(cues.values()) >= MIN_CUES
        for c in members:
            c['onset'] = t_on
            c['role'] = ('involved' if (c in early or c['anom'] or c['hit']) else 'queue') if ok else 'ignored'
        if ok:
            involved += [c for c in members if c['role'] == 'involved']
        clusters.append(dict(members=members, cues=cues, ok=ok, onset=t_on))

    forced = []
    if ACCIDENT_IDS:
        wanted = set(ACCIDENT_IDS)
        by_c = {id(c['tr']): c for c in cands}
        for tr in tracks:
            if tr.get('oid') in wanted:
                c = by_c.get(id(tr))
                forced.append(c or dict(tr=tr, s=min(tr['xy'])))
        involved = forced
    return dict(tracks=tracks, mode=mode, cands=cands, clusters=clusters, involved=involved)


def _acc_start(c):
    return c.get('onset', c['s'])


def accident_boxes_involved(frames, dt):
    res = analyze_involved(frames, dt)
    if not res['involved']:
        return {}, None
    per_frame = defaultdict(list)
    for c in res['involved']:
        st = _acc_start(c)
        for t, oi in c['tr']['oi'].items():
            if t >= st:
                per_frame[t].append(frames[t][oi]['box'])
    out = {}
    for t, bx in per_frame.items():
        box = _union(bx)
        if (box[2] - box[0]) * (box[3] - box[1]) <= MAX_BOX_AREA:
            out[t] = box
    return out, min(_acc_start(c) for c in res['involved'])


def accident_boxes_stationary(frames, dt):
    if any(o.get('head') is not None for f in frames for o in f):
        return accident_boxes_involved(frames, dt)
    return _stationary_simple(frames, dt)


# Persistent-deformation rule: abnormal cuboid pose together with sustained overlap between vehicles.
def _anomaly_runs(frames, tr, cells, dt):
    pts = {}
    for t in sorted(tr['oi']):
        o = frames[t][tr['oi'][t]]
        if o.get('head') is None or o['diag'] < DEF_MIN_DIAG:
            continue
        fl = _flow_at(cells, o['cn'], o['cls'])
        if fl is not None:
            pts[t] = (_ang_diff(o['head'], fl[0]), abs(o['asp'] - fl[1]))
    w = max(1, round(DEF_SMOOTH_S / dt / 2))
    bad = []
    for t in sorted(pts):
        near = [pts[u] for u in range(t - w, t + w + 1) if u in pts]
        if len(near) < max(2, w):
            continue
        hd = float(np.median([a for a, _ in near]))
        ad = float(np.median([b for _, b in near]))
        if hd > DEF_HEAD_TOL or (DEF_USE_ASPECT and ad > DEF_ASP_TOL):
            bad.append((t, hd, ad))
    runs = []
    for t, hd, ad in bad:
        if runs and t - runs[-1][1] <= DEF_GAP + 1:
            r = runs[-1]
            runs[-1] = [r[0], t, max(r[2], hd), max(r[3], ad)]
        else:
            runs.append([t, t, hd, ad])
    need = max(1, round(DEF_S / dt))
    badset = {t for t, _, _ in bad}
    pre = max(1, round(DEF_PRE_S / dt))
    def has_baseline(a):
        return sum(1 for u in range(a - pre, a) if u in pts and u not in badset) >= 0.6 * pre
    return [tuple(r) for r in runs if r[1] - r[0] + 1 >= need and has_baseline(r[0])]


def _persistent_overlap(frames, A, B, lo, hi):
    run, first = 0, None
    for t in range(max(lo, 0), min(hi, len(frames) - 1) + 1):
        i, j = A['oi'].get(t), B['oi'].get(t)
        ok = False
        if i is not None and j is not None:
            oa, ob = frames[t][i], frames[t][j]
            ok = ob['diag'] >= DEF_MIN_DIAG and _box_ioa(oa['box'], ob['box']) >= OVL_TOL
        if ok:
            if run == 0:
                first = t
            run += 1
            if run >= OVL_FRAMES:
                return first
        else:
            run, first = 0, None
    return None


def analyze_deformation(frames, dt):
    empty = dict(events=[], clusters=[])
    if not USE_DEFORM or not any(o.get('head') is not None for f in frames for o in f):
        return empty
    tracks, _ = tracks_for(frames, dt)
    cells = _flow_field(frames, tracks, dt)
    sp = {id(tr): track_speeds(tr, dt) for tr in tracks}
    w1 = max(1, round(1.0 / dt))

    def moved(tr, lo, hi):
        return any(v >= MOVE_REL for t, v in sp[id(tr)].items() if lo <= t <= hi)

    def at_rest(tr, lo, hi):
        return sum(1 for t, v in sp[id(tr)].items() if lo <= t <= hi and v < STOP_REL) >= max(2, w1 // 2)

    events = []
    for tr in tracks:
        for a, b, hd, ad in _anomaly_runs(frames, tr, cells, dt):
            lo, hi = a - w1 // 2, b + w1 // 2
            partners = []
            for other in tracks:
                if other is tr:
                    continue
                first = _persistent_overlap(frames, tr, other, lo, hi)
                if (first is not None and (moved(tr, lo - 2 * w1, hi + 2 * w1) or moved(other, lo - 2 * w1, hi + 2 * w1))
                        and (at_rest(tr, first, hi + 2 * w1) or at_rest(other, first, hi + 2 * w1))):
                    partners.append((first, other))
            if partners:
                events.append(dict(tr=tr, a=a, b=b, hd=hd, ad=ad, partners=partners,
                                   onset=min([a] + [f for f, _ in partners])))
    parent = list(range(len(events)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    owner = {}
    for i, ev in enumerate(events):
        for tr in [ev['tr']] + [p for _, p in ev['partners']]:
            if id(tr) in owner:
                parent[find(i)] = find(owner[id(tr)])
            else:
                owner[id(tr)] = i
    groups = defaultdict(list)
    for i, ev in enumerate(events):
        groups[find(i)].append(ev)
    clusters = []
    for evs in groups.values():
        trs = {}
        for ev in evs:
            for tr in [ev['tr']] + [p for _, p in ev['partners']]:
                trs[id(tr)] = tr
        clusters.append(dict(events=evs, tracks=list(trs.values()),
                             onset=min(ev['onset'] for ev in evs), end=max(ev['b'] for ev in evs),
                             score=sum(ev['b'] - ev['a'] + 1 for ev in evs)))
    clusters.sort(key=lambda c: -c['score'])
    clusters = clusters[:DEF_MAX_CLUSTERS]
    return dict(events=[ev for cl in clusters for ev in cl['events']], clusters=clusters)


def accident_boxes_deformation(frames, dt):
    res = analyze_deformation(frames, dt)
    per_frame = defaultdict(list)
    starts = []
    hold = round(DEF_HOLD_S / dt)
    for cl in res['clusters']:
        starts.append(cl['onset'])
        for tr in cl['tracks']:
            stops = _stationary_info(tr, dt) is not None
            last = max(tr['oi']) if stops else cl['end'] + hold
            for t, oi in tr['oi'].items():
                if cl['onset'] <= t <= last:
                    per_frame[t].append(frames[t][oi]['box'])
    out = {}
    for t, bx in per_frame.items():
        box = _union(bx)
        if (box[2] - box[0]) * (box[3] - box[1]) <= MAX_BOX_AREA:
            out[t] = box
    return out, (min(starts) if starts else None)


# Combines the rules and runs them on each continuous sequence.
def find_accidents(frames, dt):
    res, starts = {}, []
    runs = []
    if ACCIDENT_RULE in ('contact', 'both'):
        runs.append(accident_boxes(frames, dt))
    if ACCIDENT_RULE in ('stationary', 'both'):
        runs.append(accident_boxes_stationary(frames, dt))
    if ACCIDENT_RULE in ('stationary', 'both') and USE_DEFORM:
        runs.append(accident_boxes_deformation(frames, dt))
    for r, t0 in runs:
        if t0 is not None:
            starts.append(t0)
        for t, b in r.items():
            res[t] = _union([res[t], b]) if t in res else b
    return res, (min(starts) if starts else None)


SEQ_GAP_S = 2.0


def find_accidents_seg(items, dt):
    cuts = [0] + [i for i in range(1, len(items)) if items[i][0] - items[i - 1][0] > SEQ_GAP_S] + [len(items)]
    res, starts = {}, []
    for a, b in zip(cuts, cuts[1:]):
        r, t0 = find_accidents([o for _, _, o in items[a:b]], dt)
        res.update({a + t: bx for t, bx in r.items()})
        if t0 is not None:
            starts.append(a + t0)
    return res, (min(starts) if starts else None)


# Output helpers: YOLO label lines, sequence loading and summary printing.
def _line(c, box):
    x1, y1, x2, y2 = box
    return f'{c} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {x2 - x1:.6f} {y2 - y1:.6f}'


def load_sequence(set_, cam, files):
    items, n_bad = [], 0
    for p in files:
        objs, bad = load_objects(p, image_size(set_, cam, p.stem))
        n_bad += bad
        items.append((parse_ts(p.stem), p, objs))
    if all(t is not None for t, _, _ in items):
        items.sort(key=lambda x: x[0])
        dt = float(np.median(np.diff([t for t, _, _ in items]))) if len(items) > 1 else 0.1
    else:
        items.sort(key=lambda x: x[1].name)
        items = [(float(i), p, o) for i, (_, p, o) in enumerate(items)]
        dt = 0.1
    return items, (dt if dt > 0 else 0.1), n_bad


def label_groups():
    groups = defaultdict(list)
    for f in glob.glob(f'{RAW}/**/*.json', recursive=True):
        P = Path(f)
        sc = set_cam_of(P, {'_labels'})
        if sc:
            groups[sc].append(P)
    if not groups:
        raise SystemExit(f'No JSONs found in {RAW}/**/_labels/**')
    return groups


def _summary(records, n_bad, warn=True):
    per_class = Counter()
    for r in records:
        for ln in open(f"{YOLO_LABELS}/{r['set']}__{r['cam']}/{r['stem']}.txt"):
            per_class[CLASSES[int(ln.split()[0])]] += 1
    json.dump(records, open(f'{YOLO_LABELS}/index.json', 'w'))
    print('\nOriginal categories found:', dict(RAW_CATS))
    print('Instances written per class:', {c: per_class[c] for c in CLASSES if per_class[c]})
    if n_bad:
        print(f'⚠ {n_bad} objects were skipped (missing fields in JSON)')
    if warn and not per_class['accident']:
        print('⚠ No accident found. Try CONTACT_MARGIN = 0.5 or MIN_CONTACT_FRAMES = 1 and run again.')


# Conversion with the patched dev-kit converter, then adds the `accident` boxes to the label files.
import subprocess, sys
DEVKIT    = '/content/tum-traffic-dataset-dev-kit'
CONVERTER = f'{DEVKIT}/src/label_conversion/conversion_openlabel_to_yolo.py'

DEVKIT_PATCHES = [
    ('from src.utils.vis_utils import class_name_to_id_mapping',
     'from src.utils.utils import class_name_to_id_mapping'),
    ('write_line(filename, object_id, x_center, y_center, height, width)',
     'write_line(output_folder_path_labels, filename, object_id, x_center * 1920, y_center * 1200, height * 1200, width * 1920)'),
    ('            width = x_max - x_min\n            height = y_max - y_min\n',
     '            x_min, x_max = max(x_min, 0.0), min(x_max, 1.0)\n'
     '            y_min, y_max = max(y_min, 0.0), min(y_max, 1.0)\n'
     '            if x_max <= x_min or y_max <= y_min:\n'
     '                continue\n'
     '            width = x_max - x_min\n            height = y_max - y_min\n'),
    ('object_id = get_id_by_class_name(object_class)',
     'object_id = class_name_to_id_mapping.get(object_class.upper(), -1)'),
    ('            if object_id == -1:\n                continue',
     '            if object_id in (-1, 10):\n                continue'),
]

def patch_devkit_converter():
    s = open(CONVERTER).read()
    if s.startswith('# PATCHED_FOR_R0'):
        return
    for a, b in DEVKIT_PATCHES:
        if s.count(a) != 1:
            raise RuntimeError(f'Patch does not match (another version of the file?): {a[:60]!r}')
        s = s.replace(a, b)
    open(CONVERTER, 'w').write('# PATCHED_FOR_R0\n' + s)


def convert_with_devkit(only_accident_sets=True):
    patch_devkit_converter()
    shutil.rmtree(YOLO_LABELS, ignore_errors=True)
    RAW_CATS.clear()
    records, n_bad = [], 0
    groups = label_groups()
    all_sets = {s for s, _ in groups}
    pos = {s for s in all_sets if is_accident_set(s)} if only_accident_sets else all_sets
    if not pos:
        print(f'⚠ No set in {sorted(all_sets)} matches ACCIDENT_SETS={ACCIDENT_SETS}: searching for accidents in all of them.')
        pos = all_sets
    if only_accident_sets:
        print('Accident sets:', sorted(pos), '| negative sets:', sorted(all_sets - pos) or '-')
    for (set_, cam), files in sorted(groups.items()):
        out = Path(YOLO_LABELS) / f'{set_}__{cam}'
        r = subprocess.run([sys.executable, CONVERTER,
                            '--input_folder_path_labels', str(files[0].parent),
                            '--output_folder_path_labels', str(out)],
                           cwd=DEVKIT, env={**os.environ, 'PYTHONPATH': DEVKIT},
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-2000:])
            raise SystemExit(f'conversion_openlabel_to_yolo.py failed on {set_}/{cam}')
        items, dt, bad = load_sequence(set_, cam, files)
        n_bad += bad
        accs, t0 = find_accidents_seg(items, dt) if set_ in pos else ({}, None)
        for i, (t, p, objs) in enumerate(items):
            txt = out / f'{p.stem}.txt'
            if not txt.exists():
                txt.write_text('')
            if i in accs:
                with open(txt, 'a') as f:
                    f.write(_line(CLS['accident'], accs[i]) + '\n')
            records.append(dict(set=set_, cam=cam, stem=p.stem, ts=t, acc=i in accs))
        first = items[t0][1].stem if t0 is not None else ('NONE' if set_ in pos else 'negative set')
        print(f'{set_}/{cam}: {len(items)} frames, dt≈{dt:.3f}s | accident start: {first} | accident frames: {len(accs)}')
    _summary(records, n_bad, warn=only_accident_sets)


# Diagnostics: shows what the labels contain and why an accident was or was not detected.
def diagnose():
    global CONTACT_MARGIN
    lab = sorted(f for f in glob.glob(f'{RAW}/**/*.json', recursive=True) if '_labels' in Path(f).parts)
    print(f'{len(lab)} label files')
    d = json.load(open(lab[0]))
    print('JSON keys:', list(d.keys()))
    if d.get('labels'):
        print('first label (truncated):', json.dumps(d['labels'][0])[:700])
        attrs = defaultdict(Counter)
        for f in lab[:40]:
            for l in json.load(open(f)).get('labels', []):
                a = l.get('attributes', {})
                for k, v in (a.items() if isinstance(a, dict) else []):
                    attrs[k][str(v)[:30]] += 1
        for k, c in attrs.items():
            print(f'  attribute {k!r}: {dict(c.most_common(6))}')
    saved = CONTACT_MARGIN
    for (set_, cam), files in sorted(label_groups().items()):
        items, dt, _ = load_sequence(set_, cam, files)
        frames = [o for _, _, o in items]
        n_obj = sum(len(f) for f in frames)
        n3d = sum(1 for f in frames for o in f if o['c3d'] and o['size'])
        print(f'\n{set_}/{cam}: {len(items)} frames, duration {items[-1][0] - items[0][0]:.1f}s, dt≈{dt:.3f}s | objects {n_obj}, with valid 3D {n3d}')
        if n3d:
            for m in (0.0, 0.5, 1.0, 2.0):
                CONTACT_MARGIN = m
                print(f'  3D contact with margin {m} m: {sum(bool(contact_set(f)) for f in frames)} frames')
            CONTACT_MARGIN = saved
        tracks, mode = tracks_for(frames, dt)
        sp = [v for tr in tracks for v in track_speeds(tr, dt).values()]
        stops = [x for x in (stop_start(tr, dt) for tr in tracks) if x is not None]
        unit = 'm/s' if mode == '3D' else 'diagonals/s'
        print(f'  tracking: {mode} | {len(tracks)} tracks')
        if sp:
            print(f'  speeds: median {np.median(sp):.2f}, p95 {np.percentile(sp, 95):.2f}, p5 {np.percentile(sp, 5):.2f} {unit}')
        print(f'  vehicles that moved and then stopped for >= {MIN_STOP_S}s: {len(stops)}'
              + (f' | first stopped: {items[min(stops)][1].stem}' if stops else ''))
        if any(o.get('head') is not None for f in frames for o in f):
            res = analyze_involved(frames, dt)
            print(f"  --- stopped vehicles on the roadway: {len(res['cands'])} in {len(res['clusters'])} group(s) ---")
            t_first = items[0][0]
            for k, cl in enumerate(res['clusters']):
                print(f"  group {k}: cues {cl['cues']} -> {'ACCIDENT' if cl['ok'] else 'ignored'} | estimated onset at +{items[cl['onset']][0] - t_first:.1f}s")
                for c in sorted(cl['members'], key=lambda c: c['s']):
                    f = lambda v, u='': 'n/a' if v is None else f'{v:.0f}{u}' if u == '°' else f'{v:.2f}'
                    print(f"    id={c['oid']} {c['cls']:<8} pos=({c['cn'][0]:.2f},{c['cn'][1]:.2f}) stopped from +{items[c['s']][0] - t_first:.1f}s"
                          f" | moved: {c['moved']} | abrupt: {c['abrupt']} | Δorientation {f(c['head_diff'], '°')} | Δaspect {f(c['asp_diff'])}"
                          f" | overlap: {c['ovl'] is not None} | damage attribute: {c['hit']} -> {c['role']}")
            dres = analyze_deformation(frames, dt)
            print(f"  --- persistent deformation (>= {DEF_S}s + overlap): {len(dres['events'])} events in {len(dres['clusters'])} group(s) ---")
            for k, cl in enumerate(dres['clusters']):
                print(f"  group D{k}: onset +{items[cl['onset']][0] - t_first:.1f}s, deformation end +{items[cl['end']][0] - t_first:.1f}s -> ACCIDENT")
                for ev in cl['events']:
                    ps = ', '.join(f"{str(p.get('oid'))[:8]}(+{items[f][0] - t_first:.1f}s)" for f, p in ev['partners'])
                    print(f"    id={str(ev['tr'].get('oid'))[:8]} deformed +{items[ev['a']][0] - t_first:.1f}..+{items[ev['b']][0] - t_first:.1f}s"
                          f" | max Δorientation {ev['hd']:.0f}° | max Δaspect {ev['ad']:.2f} | overlapping with {ps}")

In [22]:
# Convert the labels to YOLO format and add the `accident` class.
convert_with_devkit()

a9_dataset_r00_s00/s040_camera_basler_north_16mm: 200 frames, dt≈0.080s | accident start: NONE | accident frames: 0
a9_dataset_r00_s00/s040_camera_basler_north_50mm: 198 frames, dt≈0.080s | accident start: 1617874313_523000000_s040_camera_basler_north_50mm | accident frames: 105
a9_dataset_r00_s00/s050_camera_basler_south_16mm: 200 frames, dt≈0.080s | accident start: 1617874316_505000000_s050_camera_basler_south_16mm | accident frames: 69
a9_dataset_r00_s00/s050_camera_basler_south_50mm: 200 frames, dt≈0.080s | accident start: 1617874311_220000000_s050_camera_basler_south_50mm | accident frames: 135
raw/_labels: 582 frames, dt≈0.165s | accident start: NONE | accident frames: 0

Original categories found: {'trailer': 4338, 'car': 21977, 'truck': 7872, 'van': 4651, 'other vehicles': 376, 'bus': 55, 'motorcycle': 33, 'pedestrian': 20, 'special_vehicle': 9, 'bicycle': 1}
Instances written per class: {'car': 21975, 'truck': 7872, 'trailer': 4337, 'van': 4651, 'motorcycle': 33, 'bus': 55, 'p

### If `accident frames: 0` or the accident is not where it should be
Run the cell below: it shows the structure of the labels, whether 3D boxes exist, how many frames have contact (with increasing margins) and whether any vehicles have stopped. Then look at the sequence in the verification cell.

In [23]:
# Print diagnostics about the labels and the accident rules.
diagnose()

1380 label files
JSON keys: ['image_file_name', 'labels', 'timestamp_nsecs', 'timestamp_secs']
first label (truncated): {"attributes": {"Body Color": {"value": "Gray"}, "Number of Trailers": {"value": "0"}, "Occluded": {"value": "PARTIALLY_OCCLUDED"}, "Truck Type": {"value": "Delivery Truck"}}, "category": "Truck", "id": "94b51dd1-5184-4337-ae0c-936773817037", "box3d_projected": {"bottom_left_front": [0.346025, 0.029093], "bottom_left_back": [0.341833, 0.026371], "bottom_right_back": [0.351812, 0.026371], "bottom_right_front": [0.35659, 0.029093], "top_left_front": [0.346025, 0.005452], "top_left_back": [0.341833, 0.004041], "top_right_back": [0.351812, 0.004041], "top_right_front": [0.35659, 0.005452]}}
  attribute 'Body Color': {"{'value': 'White'}": 328, "{'value': 'Gray'}": 153, "{'value': 'Other'}": 107, "{'value': 'Black'}": 96, "{'value': 'Blue'}": 79, "{'value': 'Red'}": 47}
  attribute 'Number of Trailers': {"{'value': '0'}": 700, "{'value': '1'}": 76}
  attribute 'Occluded': 

### Visual check (mandatory)
Draws a few frames with the resulting labels. The **thick red** box = `accident`, green boxes = objects. If the red box appears in frames where there is no accident (or is missing where it should be), adjust `CONTACT_MARGIN` / `MIN_CONTACT_FRAMES` in the definitions cell, run it again, then `convert_with_devkit()`.

In [ ]:
# Draw the generated labels on frames for a visual check of the `accident` boxes.
import cv2
from IPython.display import display, Image as IPImage


def _render(r, width=640):
    path = img_path(r['set'], r['cam'], r['stem'])
    if path is None:
        return None
    im = cv2.imread(path)
    if im is None:
        return None
    H, W = im.shape[:2]
    for ln in open(f"{YOLO_LABELS}/{r['set']}__{r['cam']}/{r['stem']}.txt"):
        c, xc, yc, w, h = ln.split()
        c, xc, yc, w, h = int(c), float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
        p1, p2 = (int(xc - w / 2), int(yc - h / 2)), (int(xc + w / 2), int(yc + h / 2))
        is_acc = CLASSES[c] == 'accident'
        color = (0, 0, 255) if is_acc else (0, 200, 0)
        cv2.rectangle(im, p1, p2, color, 10 if is_acc else 3)
        cv2.putText(im, CLASSES[c], (p1[0], max(p1[1] - 8, 24)), cv2.FONT_HERSHEY_SIMPLEX, 1.6, color, 3)
    return cv2.resize(im, (width, int(H * width / W)), interpolation=cv2.INTER_AREA)


def _sheet(pick, titles, cols=3, width=640, out='/content/visualization.jpg'):
    tiles, miss = [], 0
    for r, ttl in zip(pick, titles):
        t = _render(r, width)
        if t is None:
            miss += 1
            t = np.full((int(width * 0.625), width, 3), 60, np.uint8)
            cv2.putText(t, 'image missing', (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        bar = np.full((34, width, 3), 30, np.uint8)
        cv2.putText(bar, ttl[:70], (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        tiles.append(np.vstack([bar, t]))
    h = max(t.shape[0] for t in tiles)
    tiles = [np.vstack([t, np.zeros((h - t.shape[0], width, 3), np.uint8)]) if t.shape[0] < h else t for t in tiles]
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    sheet = np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])
    cv2.imwrite(out, sheet, [cv2.IMWRITE_JPEG_QUALITY, 85])
    print(f'{len(pick) - miss}/{len(pick)} images found | montage saved to {out}')
    display(IPImage(filename=out, width=1500))


def show_samples(n_acc=6, n_norm=3):
    recs = json.load(open(f'{YOLO_LABELS}/index.json'))
    acc = sorted([r for r in recs if r['acc']], key=lambda r: r['ts'])
    norm = sorted([r for r in recs if not r['acc']], key=lambda r: r['ts'])
    print(f'{len(acc)} frames with accident, {len(norm)} without')
    pick = []
    for pool, n in ((acc, n_acc), (norm, n_norm)):
        if pool:
            pick += [pool[i] for i in np.linspace(0, len(pool) - 1, min(n, len(pool))).astype(int)]
    if not pick:
        print('Nothing to show.')
        return
    _sheet(pick, [f"{r['cam'][:24]} | {'ACCIDENT' if r['acc'] else 'normal'} | {r['stem'][:20]}" for r in pick])


def show_sequence(cam_contains='south_50mm', n=12, cols=4):
    recs = json.load(open(f'{YOLO_LABELS}/index.json'))
    rs = sorted([r for r in recs if cam_contains in r['cam']], key=lambda r: r['ts'])
    if not rs:
        print('No camera contains', cam_contains, '| cameras:', sorted({r['cam'] for r in recs}))
        return
    pick = [rs[i] for i in np.linspace(0, len(rs) - 1, min(n, len(rs))).astype(int)]
    _sheet(pick, [f"+{r['ts'] - rs[0]['ts']:.1f}s | {'ACCIDENT' if r['acc'] else 'normal'} | frame {rs.index(r)}" for r in pick],
           cols, width=480, out='/content/sequence.jpg')


def show_zoom(cam_contains='south_50mm', n=8, cols=4, width=480, pad=0.6):
    recs = json.load(open(f'{YOLO_LABELS}/index.json'))
    rs = sorted([r for r in recs if cam_contains in r['cam']], key=lambda r: r['ts'])
    boxes = []
    for r in rs:
        if r['acc']:
            for ln in open(f"{YOLO_LABELS}/{r['set']}__{r['cam']}/{r['stem']}.txt"):
                c, xc, yc, w, h = ln.split()
                if CLASSES[int(c)] == 'accident':
                    xc, yc, w, h = float(xc), float(yc), float(w), float(h)
                    boxes.append((xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2))
    if not boxes:
        print('No accident area on', cam_contains, '| cameras:', sorted({r['cam'] for r in recs}))
        return
    x1, y1, x2, y2 = (float(np.median([b[i] for b in boxes])) for i in range(4))
    bw, bh = x2 - x1, y2 - y1
    x1, y1, x2, y2 = (max(0.0, x1 - pad * bw), max(0.0, y1 - pad * bh), min(1.0, x2 + pad * bw), min(1.0, y2 + pad * bh))
    pick = [rs[i] for i in np.linspace(0, len(rs) - 1, min(n, len(rs))).astype(int)]
    tiles = []
    for r in pick:
        path = img_path(r['set'], r['cam'], r['stem'])
        im = cv2.imread(path) if path else None
        if im is None:
            t = np.full((int(width * 0.75), width, 3), 60, np.uint8)
        else:
            H, W = im.shape[:2]
            for ln in open(f"{YOLO_LABELS}/{r['set']}__{r['cam']}/{r['stem']}.txt"):
                c, xc, yc, w, h = ln.split()
                c, xc, yc, w, h = int(c), float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
                acc = CLASSES[c] == 'accident'
                cv2.rectangle(im, (int(xc - w / 2), int(yc - h / 2)), (int(xc + w / 2), int(yc + h / 2)),
                              (0, 0, 255) if acc else (0, 200, 0), 6 if acc else 2)
            crop = im[int(y1 * H):int(y2 * H), int(x1 * W):int(x2 * W)]
            t = cv2.resize(crop, (width, max(1, int(crop.shape[0] * width / crop.shape[1]))), interpolation=cv2.INTER_CUBIC)
        bar = np.full((30, width, 3), 30, np.uint8)
        cv2.putText(bar, f"+{r['ts'] - rs[0]['ts']:.1f}s | {'ACCIDENT' if r['acc'] else 'normal'}", (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        tiles.append(np.vstack([bar, t]))
    h = max(t.shape[0] for t in tiles)
    tiles = [np.vstack([t, np.zeros((h - t.shape[0], width, 3), np.uint8)]) if t.shape[0] < h else t for t in tiles]
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    sheet = np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])
    cv2.imwrite('/content/zoom.jpg', sheet, [cv2.IMWRITE_JPEG_QUALITY, 90])
    display(IPImage(filename='/content/zoom.jpg', width=1500))


show_samples()
show_sequence()